In [ ]:
'''# Open loop simulation

E_grid_vec = np.zeros(Nsim)   # Wh per step
cost_vec   = np.zeros(Nsim)   # R$ per step
SoC_vec    = np.zeros(Nsim)
P_batvec   = np.zeros(Nsim)
E_curt_vec = np.zeros(Nsim)

E_load = 0.0  # Wh
E_pv   = 0.0  # Wh
E_bat  = 0.0  # Wh 


for k in range(Nsim):

    bat1.SoC, E_grid, E_curt, cost, P_bat = bat1.step_open_loop(
    SoC = bat1.SoC, P_load = load_profile[k], P_pv = pv_profile[k], tariff = tariff_sim[k], dt = dt, P_bat = 50)

    E_grid_vec[k] = E_grid
    cost_vec[k]   = cost
    P_batvec[k]   = P_bat
    SoC_vec[k]    = bat1.SoC
    E_curt_vec[k] = E_curt

    E_load += load_profile[k] * (dt) # Convert to Wh
    E_pv   += pv_profile[k] * (dt) 
    E_bat  += P_bat * (dt) 
    

cost_total = float(np.sum(cost_vec))       # R$
grid_total = np.sum(E_grid_vec)/1000.0
load_total = E_load/1000.0 
pv_total   = E_pv/1000.0
bat_total  = E_bat/1000.0
curt_total = np.sum(E_curt_vec)/1000.0


print(f'Total cost over {t_sim} hours: R${cost_total:.2f}')
print(f'Total load consumption: {load_total:.2f} kWh')
print(f'Total grid consumption over {t_sim} hours: {grid_total:.2f} kWh')
print(f'Total pv generation over {t_sim} hours: {pv_total:.2f} kWh')
print(f'Total battery contribution over {t_sim} hours: {bat_total:.2f} kWh')

#print(f'Total cost over 30 days: R${cost_total*30:.2f}')
#print(f'Total load consumption: {load_total*30:.2f} kWh')
#print(f'Total grid consumption over 30 days: {grid_total*30:.2f} kWh')
#print(f'Total pv generation over 30 days: {pv_total*30:.2f} kWh')
#print(f'Total battery contribution over 30 days: {bat_total*30:.2f} kWh')



plt.figure()
plt.plot(np.arange(0, t_sim, dt), SoC_vec)
plt.title('Battery State of Charge over Time - Open Loop')
plt.xlabel('Days')
plt.ylabel('State of Charge [%]')
plt.grid()
plt.show()



plt.figure()
plt.plot(np.arange(0, t_sim, dt), E_grid_vec/1000.0)
plt.title('Grid Energy Consumption over Time - Open Loop')
plt.xlabel('Days')
plt.ylabel('Grid Energy [kWh]')
plt.grid()
plt.show()
'''


In [ ]:
'''###### Charging Tests 

SoCk = np.zeros(Nsim)
SoCk[0] = 100 # Initial SoC


#load consumption profile
for k in range(1, Nsim):
        SoC = 0.995*SoCk[k-1] + K_ch[0]*(-load_profile[k-1] + pv_profile[k-1]) 
        SoC = np.clip(SoC, 0, 100)
        SoCk[k] = SoC

plt.figure()
plt.plot(np.arange(0, t_sim, dt), SoCk)
plt.title('Battery State of Charge over Time - Open Loop')
plt.xlabel('Time [hours]')
plt.ylabel('State of Charge [%]')
plt.grid()
plt.show()
'''

In [ ]:
########## Model
# Baterry Model
# SoC_a = previous SoC [Wh]
# loss = 0.995 == Self discharge per sample
#### SoC(k) = loss*SoC(k-1) + K1 * Pot(k-1)


A = np.array([1.0, -0.995])  # A polynomial coefficients
B = np.array([K_ch[0]])  # B polynomial coefficients 


# dt = Sampling time in hours


## GPC Matrices for Battery SoC Control

CARIMA = CARIMA(A, B, Np, Nu)
G = CARIMA["G"]
F = CARIMA["F"]


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def curva_solar_base(horas, nascer_sol=6.5, por_sol=18.0, alpha=1.8):
    """
    Curva base normalizada entre 0 e 1.
    """
    base = np.zeros_like(horas, dtype=float)
    mask = (horas >= nascer_sol) & (horas <= por_sol)

    x = (horas[mask] - nascer_sol) / (por_sol - nascer_sol)
    base[mask] = np.sin(np.pi * x) ** alpha

    return base

def adicionar_nuvens(fator, horas, rng, n_eventos, amp_min, amp_max, larg_min, larg_max):
    """
    Adiciona quedas suaves por passagem de nuvens.
    """
    out = fator.copy()

    for _ in range(n_eventos):
        centro = rng.uniform(8.0, 16.5)
        amplitude = rng.uniform(amp_min, amp_max)
        largura = rng.uniform(larg_min, larg_max)

        dip = 1.0 - amplitude * np.exp(-0.5 * ((horas - centro) / largura) ** 2)
        out *= dip

    return out

def gerar_perfil_pv(tipo="ensolarado", P_pico_kw=5.0, ts_min=15, seed=42):
    """
    Retorna um DataFrame com a potência FV ao longo de 24h.
    """
    rng = np.random.default_rng(seed)

    n = int(24 * 60 / ts_min)
    horas = np.arange(n) * ts_min / 60.0

    base = curva_solar_base(horas, nascer_sol=6.5, por_sol=18.0, alpha=1.8)

    if tipo == "ensolarado":
        clima = 0.97 + 0.01 * rng.normal(size=n)
        clima = np.clip(clima, 0.93, 1.00)
        clima = adicionar_nuvens(clima, horas, rng, n_eventos=1,
                                 amp_min=0.03, amp_max=0.08,
                                 larg_min=0.15, larg_max=0.35)

    elif tipo == "medio":
        clima = 0.62 + 0.05 * rng.normal(size=n)
        clima = np.clip(clima, 0.45, 0.75)
        clima = adicionar_nuvens(clima, horas, rng, n_eventos=3,
                                 amp_min=0.15, amp_max=0.35,
                                 larg_min=0.20, larg_max=0.60)

    elif tipo == "nublado":
        clima = 0.12 + 0.03 * rng.normal(size=n)
        clima = np.clip(clima, 0.02, 0.20)

        # pequenas aberturas no céu
        for _ in range(2):
            centro = rng.uniform(10.0, 15.0)
            largura = rng.uniform(0.20, 0.50)
            abertura = rng.uniform(0.03, 0.08) * np.exp(-0.5 * ((horas - centro) / largura) ** 2)
            clima += abertura

        clima = np.clip(clima, 0.02, 0.25)

    else:
        raise ValueError("tipo deve ser: 'ensolarado', 'medio' ou 'nublado'")

    P_pv_kw = P_pico_kw * base * clima
    P_pv_kw[base == 0] = 0.0  # garante zero à noite

    df = pd.DataFrame({
        "hora": horas,
        "P_pv_kw": P_pv_kw,
        "base_solar": base,
        "fator_clima": clima
    })

    return df

# Gerando os três dias
df_ensolarado = gerar_perfil_pv("ensolarado", P_pico_kw=5.0, ts_min=15, seed=1)
df_medio      = gerar_perfil_pv("medio",      P_pico_kw=5.0, ts_min=15, seed=2)
df_nublado    = gerar_perfil_pv("nublado",    P_pico_kw=5.0, ts_min=15, seed=3)

# Plot
plt.figure(figsize=(10,5))
plt.plot(df_ensolarado["hora"], df_ensolarado["P_pv_kw"], label="Ensolarado")
plt.plot(df_medio["hora"],      df_medio["P_pv_kw"],      label="Médio")
plt.plot(df_nublado["hora"],    df_nublado["P_pv_kw"],    label="Nublado/chuva")
plt.xlabel("Hora do dia")
plt.ylabel("Potência FV [kW]")
plt.title("Perfis de geração fotovoltaica")
plt.grid(True)
plt.legend()
plt.show()